# 06 — 비교 모델: Geptop 2.0 · ProteomeLM-L (`results/compare_baselines`)

ProteomeFit 필수 유전자 분류(서비스 모델 `results/esscls_ablation/poolesm-noseq`)와 **같은 데이터 · 같은 genus 단위 교차검증 · 같은 genus 평균**으로
비교 모델 2종을 평가한다. 둘 다 **필수 여부만** 예측하며, 결손 적합도(fitness) 예측은 하지 않는다.

| 비교 모델 | 입력 | 방식 | 출처 |
|---|---|---|---|
| **Geptop 2.0** (Wen et al., *Front. Microbiol.* 10:1236, 2019, PMID 31214154) | 프로테옴 FASTA | 학습 없음 — 기준 균주와의 상호 최고 적중(RBH) · 계통 거리 가중 합 | 이 노트북에서 재구현 (원본 코드 https://github.com/RiversDong/geptop) |
| **ProteomeLM-L** (Malbranke et al., *PNAS* 123:e2524201123, 2026, PMID 42160340) | ProteomeLM-L `hidden_states[8]` `[n, 1152]` 만 | ProteomeFit 과 같은 분류기 · 같은 폴드로 학습 | 04 노트북 제거 실험 `prot` (`results/esscls_ablation/prot`) |

**조건 맞추기**

| 항목 | ProteomeFit | Geptop 2.0 | ProteomeLM-L |
|---|---|---|---|
| 기준(참조) · 학습 정보 | 시험 · 검증 genus 밖 균주의 라벨 | **시험 genus 밖 45개 이하 균주** (원래의 DEG 37종 대신) | ProteomeFit 과 같음 |
| 필수 라벨 | `fit_mean = -inf` | 같음 | 같음 |
| 평가 샘플 | `fit_mean` 이 공란이 아닌 195,354개 | 같음 | 같음 |
| 판정 규칙 | 보정 `conf_ess` ≥ 0.5 | 점수 > 0.24 (기본 임계값) | 보정 `conf_ess` ≥ 0.5 |
| 지표 집계 | 균주별 → genus 평균 → genus 간 평균 ± 표준편차 | 같음 | 같음 |

- Geptop 의 원래 기준 집합(DEG 37종)에는 *E. coli* · *Pseudomonas* 등 우리 시험 genus 가 들어 있어 genus 제외 조건이 깨진다. 그래서 46개 학습 균주로 다시 구성했다.
- ProteomeLM-L 은 ProteomeLM 임베딩만으로 같은 조건에서 얻을 수 있는 성능이다. 04 노트북이 저장한 genus 별 지표를 그대로 읽는다.

**소요 시간 · 환경** — GPU 없이 돈다.

| 단계 | 시간 | 메모 |
|---|---|---|
| 2절 BLASTP (genus 가 다른 균주 순서쌍 1,948개) | 16코어 약 10.5시간 | 끝난 쌍은 건너뛴다. NCBI BLAST+ 필요 |
| 3절 composition vector 거리 (46개 프로테옴) | 약 10분 | `work/cv_dist.csv` 에 저장해 재사용 |

**산출물** (`results/compare_baselines/`) — `scores.tsv` (유전자별 Geptop 점수 · 판정), `metrics_by_org.csv`, `metrics_by_genus.csv`,
`compare.csv` (방법별 genus 평균 ± 표준편차 · ProteomeFit 대비 Wilcoxon p)


In [ ]:
# !pip install biopython scikit-learn scipy pandas   # + NCBI BLAST+
import os, shutil, subprocess, itertools
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
from sklearn.metrics import matthews_corrcoef, roc_auc_score, average_precision_score
from proteomefit import data

BASE_DIR = '../data/basedata'                    # 균주별 base data TSV (01 노트북)
OUT_DIR  = '../results/compare_baselines'        # 산출물
WORK_DIR = os.path.join(OUT_DIR, 'work')         # FASTA · BLAST DB · BLAST 결과 (중간 파일)
BLAST    = os.path.dirname(shutil.which('blastp') or '')   # BLAST+ 실행 파일 폴더 (PATH 에 없으면 직접 지정)
N_PROC   = max(1, os.cpu_count() - 1)            # 동시에 돌릴 단일 스레드 blastp 수
CUTOFF   = 0.24                                  # Geptop 2.0 기본 임계값 (score > CUTOFF -> 필수)
COLS     = ['mcc', 'roc_auc', 'pr_auc']          # 비교 지표
PROTEOMEFIT = '../results/esscls_ablation/poolesm-noseq/metrics_total_calib.csv'   # 서비스 모델 genus 별 지표
PLM_L       = '../results/esscls_ablation/prot/metrics_total_calib.csv'            # ProteomeLM-L 임베딩만 쓴 모델 (04 노트북)

for sub in ('faa', 'db', 'hits'):
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)
print('BLAST bin:', BLAST or 'NOT FOUND', '| blastp workers:', N_PROC)


## 1. 데이터

`data.load_samples` 로 03 노트북과 같은 라벨 · 샘플을 읽는다. FASTA 서열 이름은 **전체 유전자 번호**(base data 행 번호, 0 – 217,467)라서
BLAST 결과를 라벨 배열에 바로 색인할 수 있다. 라벨이 미상인 유전자도 프로테옴의 일부이므로 FASTA 와 점수 계산에는 넣는다.

In [ ]:
# 라벨 · 메타 [n_all = 217,468] / 평가 샘플 [n = 195,354]
D       = data.load_samples(BASE_DIR, 'fit_mean', finite_only=False)
y       = data.ess_labels(D)                                       # 평가 샘플 필수 라벨 [n] (1 = 필수)
ess_all = np.isneginf(D.yb)                                        # 전 유전자 필수 여부 [n_all] (기준 균주의 필수 목록)
org_all = D.base.org_id.to_numpy()                                 # 전 유전자 균주 [n_all]
org_ids = pd.Categorical(D.base.org_id).categories.to_numpy()      # 균주 목록 [46]
genus_of = dict(zip(D.base.org_id, D.base.org_name.str.split().str[0]))   # 균주 -> genus


def write_fasta(seqs, idx, path):
    # 단백질 서열 [m] · 전체 유전자 번호 [m] -> FASTA (">12345", 종결 기호 * 제거)
    with open(path, 'w') as f:
        for i, s in zip(idx, seqs):
            f.write(f'>{i}\n{str(s).rstrip("*")}\n')


for name in org_ids:
    idx = np.flatnonzero(org_all == name)                          # 균주의 전체 유전자 번호
    write_fasta(D.base.seq_aa.to_numpy()[idx], idx, os.path.join(WORK_DIR, 'faa', f'{name}.faa'))
print(f'genes {len(D.base):,} | eval samples {len(y):,} (essential {int(y.sum()):,}) | '
      f'strains {len(org_ids)} | genera {len(set(genus_of.values()))}')


## 2. Geptop 2.0 — BLASTP (서로 다른 genus 의 균주 순서쌍)

Geptop 은 대상 → 기준, 기준 → 대상 두 방향의 BLASTP 최고 적중이 서로를 가리키면 직교상동체(상호 최고 적중, reciprocal best hit)로 본다.
시험 genus 의 균주는 기준에서 빠지므로 같은 genus 안의 쌍은 필요 없다 (46 × 45 = 2,070 중 1,948쌍). 질의별 상위 5개 적중을 받아 첫 줄(E-value 최소)을 최고 적중으로 쓴다.
단일 스레드 blastp 여러 개를 동시에 돌리는 편이 다중 스레드 하나보다 약 3배 빠르다.

In [ ]:
def blast_exe(blast_dir, name):
    # BLAST+ 실행 파일 경로 ('blastp' -> '.../blastp' 또는 '.../blastp.exe')
    return shutil.which(name, path=blast_dir) or os.path.join(blast_dir, name)


def make_db(blast_dir, faa, db):
    # 균주 FASTA -> 단백질 BLAST DB (이미 있으면 건너뜀)
    if not os.path.exists(db + '.pin'):
        subprocess.run([blast_exe(blast_dir, 'makeblastdb'), '-in', faa, '-dbtype', 'prot', '-out', db],
                       check=True, stdout=subprocess.DEVNULL)


def run_pair(blast_dir, work_dir, q, s):
    # 질의 균주 q 전 단백질 -> 균주 s DB, 상위 5개 적중 표 (끝난 쌍은 건너뜀, .part -> 완료 시 이름 변경)
    out = os.path.join(work_dir, 'hits', f'{q}__{s}.tsv')
    if not os.path.exists(out):
        subprocess.run([blast_exe(blast_dir, 'blastp'), '-query', os.path.join(work_dir, 'faa', f'{q}.faa'),
                        '-db', os.path.join(work_dir, 'db', s), '-outfmt', '6 qseqid sseqid evalue bitscore',
                        '-max_target_seqs', '5', '-evalue', '10', '-num_threads', '1', '-out', out + '.part'], check=True)
        os.replace(out + '.part', out)
    return out


for name in org_ids:
    make_db(BLAST, os.path.join(WORK_DIR, 'faa', f'{name}.faa'), os.path.join(WORK_DIR, 'db', name))

PAIRS = [(q, s) for q, s in itertools.permutations(org_ids, 2) if genus_of[q] != genus_of[s]]   # 1,948쌍
with ThreadPoolExecutor(N_PROC) as ex:                             # subprocess 대기는 GIL 을 놓으므로 스레드로 충분
    for k, _ in enumerate(ex.map(lambda p: run_pair(BLAST, WORK_DIR, *p), PAIRS), 1):
        if k % 100 == 0 or k == len(PAIRS):
            print(f'BLAST pairs {k}/{len(PAIRS)}')


## 3. Geptop 2.0 — 계통 거리 (composition vector, $k = 6$)

균주 프로테옴의 6-mer 빈도에서 5-mer · 4-mer 로 만든 마르코프 기대 빈도 $p_0$ 를 빼 정규화한 벡터
$a(\alpha) = (f(\alpha) - p_0(\alpha)) / p_0(\alpha)$ 를 만들고, 두 벡터의 코사인으로 거리 $D = (1 - \cos) / 2$ 를 구한다.
- 예: 6-mer `MKVLAA` 가 120번 나오고 기대 빈도 $p_0$ 가 80이면 $a = (120 - 80) / 80 = 0.5$ (기대보다 50% 많다)
- 배포 코드 `geptop2.py` 의 `Distance()` 는 분모의 두 번째 제곱합에 공통 k-mer 곱을 더하는데, 원본과 같은 점수를 내기 위해 그대로 따른다
  (원본 함수와 값이 같음을 확인했다 — Kangiella–Synechococcus 거리 0.4961469597605, 벡터 원소 최대 차이 0)

In [ ]:
# Geptop 아미노산 부호 (B -> D, U -> C, X -> G, Z -> E, J -> I), 그 외 문자가 들어간 k-mer 는 버린다
AA  = {c: i for i, c in enumerate('ACDEFGHIKLMNPQRSTVWY')} | {'B': 2, 'U': 1, 'X': 5, 'Z': 3, 'J': 7}
LUT = np.full(256, -1, np.int64)
for c, i in AA.items():
    LUT[ord(c)] = i


def kmer_counts(seqs, k, lut):
    # 단백질 서열 목록 -> (k-mer 부호 정렬 [u], 개수 [u]); 부호 = 20진수 (예: 'AC' -> 0*20 + 1 = 1)
    codes = []
    for s in seqs:
        x = lut[np.frombuffer(str(s).encode(), np.uint8)]          # 잔기 부호 [L] (-1 = 무효)
        if len(x) >= k:
            win = np.lib.stride_tricks.sliding_window_view(x, k)   # [L-k+1, k]
            codes.append(win[(win >= 0).all(1)] @ (20 ** np.arange(k - 1, -1, -1)))
    return np.unique(np.concatenate(codes), return_counts=True)


def lookup(keys, vals, q):
    # 희소 사전 조회: 정렬된 keys [u] · vals [u] · 질의 q [m] -> (있음 [m], 값 [m], 없으면 0)
    pos = np.clip(np.searchsorted(keys, q), 0, len(keys) - 1)
    hit = keys[pos] == q
    return hit, np.where(hit, vals[pos], 0)


def composition_vector(seqs, lut):
    # 프로테옴 -> composition vector (6-mer 부호 [u], 값 [u]); geptop2.py CompositionVector 와 같은 값
    (k0, c0), (k1, c1), (k2, c2) = (kmer_counts(seqs, k, lut) for k in (6, 5, 4))
    s0, s1, s2 = c0.sum(), c1.sum(), c2.sum()
    n1, a = np.repeat(k1, 20), np.tile(np.arange(20), len(k1))     # 5-mer x 다음 아미노산 [20u]
    n0, n2, n3 = n1 * 20 + a, n1 % 20 ** 4 * 20 + a, n1 % 20 ** 4  # 6-mer · 뒤쪽 5-mer · 가운데 4-mer
    ok, c_n2 = lookup(k1, c1, n2)                                  # 뒤쪽 5-mer 가 있는 경우만 쓴다
    _, c_n3 = lookup(k2, c2, n3[ok])
    n0 = n0[ok]
    p0 = 1.0 * np.repeat(c1, 20)[ok] * c_n2[ok] * s0 * s2 / c_n3 / s1 / s1   # 마르코프 기대 빈도
    has0, c_n0 = lookup(k0, c0, n0)
    v = np.where(has0, (c_n0 - p0) / p0, -1.0)                     # 관측되지 않은 6-mer = -1
    order = np.argsort(n0)
    return n0[order], v[order]


def cv_distance(cv1, cv2):
    # composition vector 2개 -> 거리 (0 = 같음); geptop2.py Distance 와 같은 식
    (a, va), (b, vb) = cv1, cv2
    _, ia, ib = np.intersect1d(a, b, assume_unique=True, return_indices=True)
    prod = (va[ia] * vb[ib]).sum()                                 # 공통 k-mer 곱의 합
    only_b = np.ones(len(b), bool); only_b[ib] = False
    return (1 - prod / np.sqrt((va ** 2).sum() * (prod + (vb[only_b] ** 2).sum()))) / 2


dist_path = os.path.join(WORK_DIR, 'cv_dist.csv')
if os.path.exists(dist_path):
    DIST = pd.read_csv(dist_path, index_col=0)
else:
    CVS  = {n: composition_vector(D.base.seq_aa.to_numpy()[org_all == n], LUT) for n in org_ids}
    DIST = pd.DataFrame({b: {a: 0.0 if a == b else cv_distance(CVS[a], CVS[b]) for a in org_ids} for b in org_ids})
    DIST.to_csv(dist_path)
    del CVS
print('CV distance', DIST.shape, f'off-diagonal {DIST.values[DIST.values > 0].min():.3f} - {DIST.values.max():.3f}')


## 4. Geptop 2.0 — 점수

대상 균주마다 **다른 genus 의** 기준 균주 각각에서 상호 최고 적중을 찾고, 그 짝이 기준 균주에서 필수이면 $1/D$ 를 더한다.
대상 균주 전 유전자에서 min-max 로 0–1 점수를 만들고, 0.24 보다 크면 필수로 판정한다.

$$S_i = \mathrm{minmax}_i\Big(\sum_{j \in \mathrm{ref}} \mathbb{1}\big[\mathrm{RBH}_j(i)\ \text{is essential in}\ j\big] \cdot w_j\Big), \qquad w_j = 1 / D_j$$

- 예: 짝이 필수인 기준 균주가 거리 0.40 · 0.50 인 두 균주이면 합 = 2.5 + 2.0 = 4.5, 균주 내 최댓값이 30 이면 점수 = 0.15 → 비필수

In [ ]:
def best_hits(path):
    # BLAST 표 -> 질의 유전자 번호 -> 최고 적중 유전자 번호 Series (질의별 첫 줄 = E-value 최소)
    t = pd.read_csv(path, sep='\t', header=None, usecols=[0, 1], names=['q', 's']).drop_duplicates('q')
    return pd.Series(t.s.to_numpy(), index=t.q.to_numpy())


def geptop_scores(org_ids, genus_of, dist, ess_all, gene_org, hits_dir):
    # 전 유전자 Geptop 점수 [n_all] (균주별 min-max 0-1). gene_org = 유전자별 균주 [n_all]
    S = np.zeros(len(ess_all))
    for a in org_ids:
        for b in org_ids:
            if genus_of[a] == genus_of[b]:
                continue
            fwd = best_hits(os.path.join(hits_dir, f'{a}__{b}.tsv'))      # a 유전자 -> b 최고 적중
            rev = best_hits(os.path.join(hits_dir, f'{b}__{a}.tsv'))      # b 유전자 -> a 최고 적중
            q, s = fwd.index.to_numpy(), fwd.to_numpy()
            rbh = rev.reindex(s).to_numpy() == q                          # 상호 최고 적중 [m]
            d = dist.loc[a, b]
            S[q[rbh & ess_all[s]]] += 100.0 if d == 0 else 1.0 / d        # 짝이 필수이면 1/D 가산
        m = gene_org == a
        S[m] = (S[m] - S[m].min()) / (S[m].max() - S[m].min())            # 균주 내 min-max
    return S


GEPTOP = geptop_scores(org_ids, genus_of, DIST, ess_all, org_all, os.path.join(WORK_DIR, 'hits'))   # [n_all]
print(f'Geptop: predicted essential {int((GEPTOP > CUTOFF).sum()):,} / {len(GEPTOP):,} genes (cutoff {CUTOFF})')
pd.DataFrame({'org_id': org_all, 'gene_id': D.base.gene_id, 'essential': ess_all.astype(int),
              'geptop_score': GEPTOP.round(4), 'geptop_call': (GEPTOP > CUTOFF).astype(int)}
             ).to_csv(os.path.join(OUT_DIR, 'scores.tsv'), sep='\t', index=False)


## 5. 비교

균주별 지표 → genus 평균 → genus 간 평균 ± 표준편차. 필수 · 비필수가 모두 있는 균주만 쓴다 (ProteomeFit 과 같은 규약).
ProteomeFit · ProteomeLM-L 수치는 03 · 04 노트북이 저장한 genus 별 지표를 그대로 읽는다. 세 모델 모두 시험 genus 를 기준 · 학습에서 보지 못했다.

- `mcc` 는 각 모델의 판정 규칙(ProteomeFit · ProteomeLM-L 보정 확률 ≥ 0.5, Geptop 점수 > 0.24)에서 구한 값이다
- `roc_auc` · `pr_auc` 는 임계값과 무관하다
- 필수 유전자가 없는 genus (`Magnetospirillum` · `Paraburkholderia`) 는 분류 지표가 정의되지 않아 빠진다 (26개 genus)
- `p_*` 는 genus 별 지표로 ProteomeFit 과 짝지은 Wilcoxon 부호 순위 검정 p 값이다


In [ ]:
def genus_metrics(org, genus_of, y, score, call, cols=COLS):
    # 균주 [n] · 라벨 [n] · 점수 [n] · 판정 [n] -> (균주별 지표 [n_org, *], genus 평균 [n_genus, 3])
    rows = []
    for o in pd.unique(org):
        m = org == o
        if not 0 < y[m].sum() < m.sum():                               # 필수 · 비필수가 모두 있는 균주만
            continue
        rows.append(dict(org_id=o, genus=genus_of[o], n=int(m.sum()), n_pos=int(y[m].sum()),
                         mcc=matthews_corrcoef(y[m], call[m]), roc_auc=roc_auc_score(y[m], score[m]),
                         pr_auc=average_precision_score(y[m], score[m])))
    om = pd.DataFrame(rows)
    return om, om.groupby('genus')[cols].mean()


def load_genus_metrics(path, cols=COLS):
    # metrics_total*.csv (03 · 04 노트북) -> genus 별 지표 [n_genus, 3] (mean · median · std 요약 줄 제외)
    t = pd.read_csv(path)
    return t[pd.to_numeric(t.fold, errors='coerce').notna()].dropna(subset=cols).set_index('genus')[cols]


def summary(gm_by_method, ref, cols=COLS):
    # {방법: genus 별 지표} -> 평균 ± 표준편차 표 + 기준 방법 ref 와의 짝 Wilcoxon p
    out = {}
    for name, gm in gm_by_method.items():
        row = {c: f'{gm[c].mean():.3f} ± {gm[c].std():.3f}' for c in cols} | {'n_genus': len(gm)}
        if name != ref:
            for c in cols:
                a, b = gm_by_method[ref][c].align(gm[c], join='inner')
                row[f'p_{c}'] = f'{wilcoxon(a, b).pvalue:.1e}'
        out[name] = row
    return pd.DataFrame(out).T


# Geptop 균주별 · genus 별 지표 (평가 샘플 = fit_mean 이 공란이 아닌 유전자)
om_gep, gm_gep = genus_metrics(D.org, genus_of, y, GEPTOP[D.keep], (GEPTOP[D.keep] > CUTOFF).astype(int))
om_gep.assign(method='Geptop 2.0').to_csv(os.path.join(OUT_DIR, 'metrics_by_org.csv'), index=False)

GM = {'ProteomeFit': load_genus_metrics(PROTEOMEFIT), 'Geptop 2.0': gm_gep, 'ProteomeLM-L': load_genus_metrics(PLM_L)}
pd.concat(GM, names=['method']).to_csv(os.path.join(OUT_DIR, 'metrics_by_genus.csv'))

tab = summary(GM, 'ProteomeFit')                                       # [3 methods, mcc · roc_auc · pr_auc · n_genus · p_*]
tab.to_csv(os.path.join(OUT_DIR, 'compare.csv'))
print(tab.to_string())
